In [130]:
import pandas as pd
import boto3
import io

Cria uma seção no boto3, passa o serviço (s3)

In [131]:
session = boto3.Session(
    profile_name='559050252078_AdministratorAccess'
)
s3 = session.client('s3')

Lê o objeto do s3, BytesIO transforma os dados em bytes, para um 'arquivo' para ser lido pelo pandas.

In [134]:
objeto = s3.get_object(
                        Bucket='analisedados', 
                        Key='ocorrencias_aeronauticas.csv'
)
dados = objeto['Body'].read()
csv = io.BytesIO(dados)
df = pd.read_csv(csv, sep=';')
df

,Numero_da_Ocorrencia,Numero_da_Ficha,Operador_Padronizado,Classificacao_da_Ocorrencia,Data_da_Ocorrencia,Hora_da_Ocorrencia,Municipio,UF,Regiao,Descricao_do_Tipo,...,Lesoes_Desconhecidas_Tripulantes,Lesoes_Desconhecidas_Passageiros,Lesoes_Desconhecidas_Terceiros,Modelo,CLS,Tipo_ICAO,PMD,Numero_de_Assentos,Nome_do_Fabricante,PSSO
0,4748,/2001,SEC.SEG.PUBLICA-SP-DEPATRI-SAT,Acidente,2001-07-14,NaN,NaN,Indeterminado,NaN,NaN,...,0.0,0.0,0.0,AS 350 B2,H1T,AS50,2100.0,6.0,EUROCOPTER FRANCE,falso
1,4747,/2001,CIRANO MARCOS DE ARAUJO,Acidente,2001-07-09,NaN,NaN,Indeterminado,NaN,NaN,...,0.0,0.0,0.0,T210N,L1P,C210,1814.0,6.0,CESSNA AIRCRAFT,verdadeiro
2,4746,/2001,AEROCLUBE DE SANTIAGO,Acidente,2001-06-29,NaN,NaN,Indeterminado,NaN,NaN,...,0.0,0.0,0.0,PA-28-180,L1P,P28A,1111.0,4.0,PIPER AIRCRAFT,verdadeiro
3,4745,/2001,HELIRIO TAXI AEREO LTDA,Acidente,2001-06-25,NaN,NaN,Indeterminado,NaN,NaN,...,0.0,0.0,0.0,369D,H1T,H500,1519.0,5.0,HUGHES HELICOPTER,verdadeiro
4,4744,/2001,ZAKI AEROTAXI LTDA,Acidente,2001-06-23,NaN,NaN,Indeterminado,NaN,NaN,...,0.0,0.0,0.0,EMB-810D,L2P,PA34,2155.0,7.0,EMBRAER,verdadeiro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5016,4961,0346/2014,HORUS ESCOLA DE AVIACAO CIVIL LTDA,Acidente,2014-02-15,22:00:00,BRAGANÇA,PA,Norte,MÉDICO,...,0.0,0.0,0.0,R44,H1P,R44,1088.0,4.0,ROBINSON HELICOPTER,verdadeiro
5017,4752,/2001,PENTA PENA TRANSPORTES AEREOS S/A,Acidente,2001-08-13,NaN,NaN,Indeterminado,NaN,NaN,...,0.0,0.0,0.0,208B,L1T,C208,3969.0,11.0,CESSNA AIRCRAFT,verdadeiro
5018,4751,/2001,TOCANTINS TAXI AEREO LTDA,Acidente,2001-08-11,NaN,NaN,Indeterminado,NaN,NaN,...,0.0,0.0,0.0,EMB-721C,L1P,P32R,1633.0,7.0,NEIVA,verdadeiro
5019,4750,/2001,SOCIEDADE ASAS DE SOCORRO,Acidente,2001-08-03,NaN,NaN,Indeterminado,NaN,NaN,...,0.0,0.0,0.0,U206G,L1P,C206,1633.0,6.0,CESSNA AIRCRAFT,verdadeiro


Com o dataframe criado podemos executar as análises.

### Análises

##### Pergunta 1: Qual é o número total de ocorrências por tipo de operação  onde houve ao menos uma lesão fatal ou grave?

Cria o filtro para cada linha do dataframe

In [ ]:
lesoes = (
    (df['Lesoes_Fatais_Tripulantes'] > 0) |
    (df['Lesoes_Fatais_Passageiros'] > 0) |
    (df['Lesoes_Fatais_Terceiros'] > 0) |
    (df['Lesoes_Graves_Tripulantes'] > 0) |
    (df['Lesoes_Graves_Passageiros'] > 0) |
    (df['Lesoes_Graves_Terceiros'] > 0)
)

Cria o dataframe "operacoes" com o filtro, mostra a quatidade de acidentes por operação e ordena decrescentemente.

In [135]:
operacoes = df.loc[lesoes].groupby('Operacao').size()
resposta1 = operacoes.sort_values(ascending=False).to_dict()
resposta1

{'Voo Privado': 648,
 'Operação Agrícola': 183,
 'Táxi Aéreo': 109,
 'Voo de Instrução': 78,
 'Operação Especializada': 36,
 'Voo Experimental': 25,
 'Operação Pública': 24,
 'Voo Regular': 22,
 'Desconhecida': 3,
 'Operação Policial': 3,
 'Operação Militar': 1,
 'Voo não regular': 1}

##### Pergunta 2: Qual é a proporção de ocorrências com danos 'Destruída', que aconteceram na fase de pouso, para aeronaves com mais de 4 assentos?

Cria filtro para o número de assentos e para quando a Aeronave foi destruída, e cria o dataframe filtrado.

In [118]:
filtro_fase_tamanho = (df['Numero_de_Assentos'] > 4) & (df['Fase_da_Operacao'] == 'Pouso')
danos = df['Danos_a_Aeronave'].isin(['Destruída'])
ocorrencias_filtradas = df.loc[filtro_fase_tamanho & danos]

Guarda a quantidade de ocorrencias com aviões com mais de 4 assentos, e a quantidade de ocorrências com 4 assentos e com danos a aeronave sendo destruída. 

In [119]:
qtd_ocorrencias = df.loc[filtro_fase_tamanho].shape[0]
qtd_danos = ocorrencias_filtradas.shape[0]

Calcula a proporção de aviões destruídos.

In [136]:
resposta2 = qtd_danos / qtd_ocorrencias
resposta2

0.01910828025477707

##### Pergunta 3: Proporção de ocorrências que aconteceram no ano de 2023, onde a descrição do tipo de ocorrência contém a palavra 'colisão'?

Transforma a coluna ""Data_da_Ocorrencia" em datetime, e filtra pelo ano de 2023"

In [ ]:
df['Data_da_Ocorrencia'] = pd.to_datetime(df['Data_da_Ocorrencia'])
df_2023 = df[df['Data_da_Ocorrencia'].dt.year == 2023]

Filtra somente as linhas que contem colisão na coluna "Descrição_do_Tipo".

In [122]:
df_col_2023 = df_2023[df_2023['Descricao_do_Tipo'].str.contains(r'colis[aã]o' , case=False, regex=True)]

Guarda quantidade de ocorênciasa e colisões.

In [123]:
qtd_ocorrencias = df_2023.shape[0]
qtd_colisoes = df_col_2023.shape[0]

Calcula a proporção.

In [137]:
resposta3 = qtd_colisoes / qtd_ocorrencias
resposta3

0.07961783439490445

Escreve as respostas em um arquivo txt.

In [128]:
with open("respostas.txt", "a") as r:
    r.write("-- Pergunta 1: \n\n")
    r.write('Qual é o número total de ocorrências por tipo de operação \nonde houve ao menos uma lesão fatal ou grave?\n\n')
    r.write("Resposta:\n\n")

    for key, value in resposta1.items():
        r.write(f"{key} : {value} \n")
    r.write('\n')
    r.write("-- Pergunta 2: \n\n")
    r.write('Qual é a proporção de ocorrências com danos \'Destruída\', \nque aconteceram na fase de pouso, para aeronaves com mais de 4 assentos?\n\n')
    r.write("Resposta:\n\n")
    r.write(f'Durante a fase de pouso {resposta2*100:.2f}% das ocorências \nacarretaram na destruição da aeronave.\n\n')

    r.write("-- Pergunta 3: \n\n")
    r.write('Proporção de ocorrências que aconteceram no ano de 2023, \nonde a descrição do tipo de ocorrência contém a palavra \'colisão\'?\n\n')
    r.write("Resposta:\n\n")
    r.write(f'{resposta3*100:.2f}% das ocorrências de 2023 contêm a palavra \'colisão\'')